# Notebook 01 — Setup dos dados

Carrega os dados já processados do projeto anterior e prepara as três partições
que conformal prediction precisa: treino, calibração e teste.

A divisão é diferente do projeto principal: aqui o conjunto de teste original
vira o teste mesmo, mas o treino original é dividido em treino-puro e calibração.
A calibração é onde o conformal aprende os scores de não-conformidade.

## 1. Configuração

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import pickle
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')

# pasta nova só para esse projeto de conformal
SAIDA = DRIVE / 'conformal_violence'
SAIDA.mkdir(exist_ok=True)

RANDOM_STATE = 42
ALVOS = ['y_fisic', 'y_psico', 'y_sexu']

## 2. Carregando os dados já processados

Mesmos arquivos do projeto anterior.

In [ ]:
X_train_orig = load_npz(DRIVE / 'X_train.npz')
X_test = load_npz(DRIVE / 'X_test.npz')
y_train_orig = pd.read_parquet(DRIVE / 'y_train.parquet')
y_test = pd.read_parquet(DRIVE / 'y_test.parquet')

with open(DRIVE / 'feature_names.json') as f:
    feature_names = json.load(f)

print(f'Treino original: {X_train_orig.shape[0]:,} linhas')
print(f'Teste:           {X_test.shape[0]:,} linhas')

## 3. Removendo casos com alvo faltante

Cerca de 3% das notificações têm pelo menos um dos três tipos de violência sem
classificação no SINAN. Para garantir uma comparação justa entre os três
desfechos (todos os modelos usam exatamente o mesmo conjunto), removo essas
linhas. A perda é pequena e o ganho em clareza metodológica compensa.

In [ ]:
# treino: mantém só linhas com TODOS os 3 alvos preenchidos
mask_train = y_train_orig[ALVOS].notna().all(axis=1).values
X_train_full = X_train_orig[mask_train]
y_train_full = y_train_orig[mask_train].reset_index(drop=True)

# teste: mesma regra
mask_test = y_test[ALVOS].notna().all(axis=1).values
X_test_full = X_test[mask_test]
y_test_full = y_test[mask_test].reset_index(drop=True)

# converte float32 para int (agora que não tem mais NaN)
for alvo in ALVOS:
    y_train_full[alvo] = y_train_full[alvo].astype(int)
    y_test_full[alvo] = y_test_full[alvo].astype(int)

print(f'Treino após limpeza: {X_train_full.shape[0]:,} ({mask_train.mean():.1%} retidas)')
print(f'Teste após limpeza:  {X_test_full.shape[0]:,} ({mask_test.mean():.1%} retidas)')

## 4. Dividindo o treino em treino-puro e calibração

Conformal prediction precisa de um conjunto separado, nunca visto pelo modelo
durante o treino. Vou usar 70% para treino-puro e 30% para calibração. A
proporção é maior que num split padrão de validação porque o Locart subdivide
ainda mais a calibração internamente, e precisa de dados suficientes em cada folha.

Estratifico pelo alvo de violência sexual (o mais raro) para garantir que a
calibração tenha positivos de todos os tipos.

In [ ]:
idx_all = np.arange(X_train_full.shape[0])

idx_train, idx_cal = train_test_split(
    idx_all,
    test_size=0.30,
    stratify=y_train_full['y_sexu'].values,
    random_state=RANDOM_STATE
)

print(f'Treino:    {len(idx_train):,}')
print(f'Calibração: {len(idx_cal):,}')

## 5. Reduzindo o treino para experimentação

Treinar logística em 1.2 milhões de linhas no Colab grátis funciona, mas demora.
Para iterar rápido na validação da abordagem, uso 80 mil para o treino. Quando
o pipeline estiver validado, basta voltar para o conjunto inteiro.

Calibração e teste ficam intocados.

In [ ]:
N_TREINO = 80_000

if len(idx_train) > N_TREINO:
    idx_train, _ = train_test_split(
        idx_train,
        train_size=N_TREINO,
        stratify=y_train_full.iloc[idx_train]['y_sexu'].values,
        random_state=RANDOM_STATE
    )

print(f'Treino final: {len(idx_train):,}')

## 6. Montando os splits

Converto a matriz esparsa para densa só nos subconjuntos que vou usar. Como
reduzi o treino para 80 mil, cabe tranquilo na memória.

In [ ]:
X_train = np.asarray(X_train_full[idx_train].todense())
X_cal = np.asarray(X_train_full[idx_cal].todense())
X_test_dense = np.asarray(X_test_full.todense())

y_train = y_train_full.iloc[idx_train].reset_index(drop=True)
y_cal = y_train_full.iloc[idx_cal].reset_index(drop=True)

# confere as prevalências em cada split
print('Prevalências:')
print(f'{"alvo":<12} {"treino":<10} {"calibração":<12} {"teste":<10}')
for alvo in ALVOS:
    p_tr = y_train[alvo].mean()
    p_cal = y_cal[alvo].mean()
    p_te = y_test_full[alvo].mean()
    print(f'{alvo:<12} {p_tr:<10.1%} {p_cal:<12.1%} {p_te:<10.1%}')

## 7. Salvando

Guardo os 3 splits e as colunas para os próximos notebooks.

In [ ]:
dados = {
    'X_train': X_train, 'y_train': y_train,
    'X_cal': X_cal, 'y_cal': y_cal,
    'X_test': X_test_dense, 'y_test': y_test_full,
    'feature_names': feature_names,
    'alvos': ALVOS,
}

with open(SAIDA / 'dados_conformal.pkl', 'wb') as f:
    pickle.dump(dados, f)

print(f'Salvo em {SAIDA / "dados_conformal.pkl"}')
print(f'Tamanho do arquivo: {(SAIDA / "dados_conformal.pkl").stat().st_size / 1024 / 1024:.1f} MB')

## Próximo passo

Notebook 02: roda MAPIE (LAC e APS) como baselines de conformal prediction nos
3 desfechos.